In [1]:

# 셀1 - 데이터 로드
import os, numpy as np, time
os.chdir('/home/xilinx/jupyter_notebooks/mobilenet')
W_CONV_LAYER = 1525656
B_CONV_LAYER = 17056
FC_LAYER     = 360000
B_FC_LAYER   = 1000
IMAGE_SIZE   = 224*224*3

def load_dat(fp):
    t0 = time.time()
    with open(fp,'r') as f: a = np.fromstring(f.read(), sep=' ', dtype=np.int32)
    print(f"  {fp}: {len(a):,} ints in {time.time()-t0:.1f}s")
    return a

image_data   = load_dat('image_int.dat')[:IMAGE_SIZE]
all_w        = load_dat('weights.dat')
weights_CONV = all_w[:W_CONV_LAYER]
weights_FC   = all_w[W_CONV_LAYER:W_CONV_LAYER+FC_LAYER]
all_b        = load_dat('bias.dat')
bias_CONV    = all_b[:B_CONV_LAYER]
bias_FC      = all_b[B_CONV_LAYER:B_CONV_LAYER+B_FC_LAYER]

tile_3x3   = np.fromfile('tile_3x3.bin',   dtype=np.int32)
tile_convs = np.fromfile('tile_convs.bin', dtype=np.int32)
tile_avg   = np.fromfile('tile_avg.bin',   dtype=np.int32)
tile_fc    = np.fromfile('tile_fc.bin',    dtype=np.int32)
info_3x3   = np.fromfile('info_3x3.bin',   dtype=np.int32)
info_convs = np.fromfile('info_convs.bin', dtype=np.int32)
info_avg   = np.fromfile('info_avg.bin',   dtype=np.int32)
info_fc    = np.fromfile('info_fc.bin',    dtype=np.int32)
print("Data loaded")

  image_int.dat: 150,528 ints in 0.1s
  weights.dat: 1,885,656 ints in 1.2s
  bias.dat: 57,056 ints in 0.0s
Data loaded


In [2]:

# 셀 2 - overlay
from pynq import Overlay, MMIO, allocate
ol = Overlay('mobilenet.bit')
print("Bitstream loaded.")
ctrl = MMIO(0x40000000, 0x10000)
print(f"ap_ctrl reg = 0x{ctrl.read(0):08x}")


Bitstream loaded.
ap_ctrl reg = 0x00000004


In [3]:

# 셀 3 - 12 buffers
def alloc_and_copy(src):
    buf = allocate(shape=src.shape, dtype=np.int32)
    np.copyto(buf, src)
    buf.flush()
    return buf

t0 = time.time()
buf_w_conv     = alloc_and_copy(weights_CONV)
buf_w_fc       = alloc_and_copy(weights_FC)
buf_b_conv     = alloc_and_copy(bias_CONV)
buf_b_fc       = alloc_and_copy(bias_FC)
buf_tile_3x3   = alloc_and_copy(tile_3x3)
buf_tile_convs = alloc_and_copy(tile_convs)
buf_tile_avg   = alloc_and_copy(tile_avg)
buf_tile_fc    = alloc_and_copy(tile_fc)
buf_info_3x3   = alloc_and_copy(info_3x3)
buf_info_convs = alloc_and_copy(info_convs)
buf_info_avg   = alloc_and_copy(info_avg)
buf_info_fc    = alloc_and_copy(info_fc)
print(f"Allocated 12 buffers in {time.time()-t0:.1f}s")


Allocated 12 buffers in 0.1s


In [4]:

# 셀 4 - 4 pointers
data_mmio = MMIO(0x40010000, 0x10000)
data_mmio.write(0x10, buf_w_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x14, (buf_w_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x1C, buf_b_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x20, (buf_b_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x28, buf_w_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x2C, (buf_w_fc.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x34, buf_b_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x38, (buf_b_fc.physical_address >> 32) & 0xFFFFFFFF)
print('4 pointers written.')


4 pointers written.


In [5]:

# 셀 5 - DMA channels + helpers + Layer 0
ip_in  = ol.axi_dma_0.sendchannel
ip_out = ol.axi_dma_1.recvchannel
res_rd = ol.axi_dma_2.sendchannel
res_wr = ol.axi_dma_2.recvchannel
get_ipython().run_line_magic('run', '-i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers.py')
get_ipython().run_line_magic('run', '-i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers_v2.py')
get_ipython().run_line_magic('run', '-i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers_v3.py')
get_ipython().run_line_magic('run', '-i /home/xilinx/jupyter_notebooks/mobilenet/python/run_layer0.py')
print("Helpers + Layer 0 loaded")


helpers.py loaded.
  ctrl @ 0x40000000, data @ 0x40010000
  ap_idle = 1
helpers_v2.py loaded.
  Functions: dram_2_stream_3x3_type2, dram_2_stream_1x1_exp0/1,
             stream_2_dram_3x3_exp0/1, stream_2_dram_1x1_exp0,
             tile/info offset helpers, conv_batch_relu_pe
helpers_v3.py loaded (SDK-correct port).
  dram_2_stream_3x3_v3, dram_2_stream_1x1_v3, dram_2_stream_array_v3
  stream_2_dram_3x3_v3, stream_2_dram_1x1_v3, stream_2_dram_array_v3
  call_ip_pe (universal IP call helper)
=== Packing input (this may take 30-60s in pure Python) ===
  packed input: 1,382,400 ints  (8.7s)
  buf_input phys = 0x18c00000
  per-PE output buffer: 100,352 ints x 4 PE

=== Calling IP per PE ===
  ap_idle before = 1
  PE 0: tile=0x15872000, info=0x15878000 ... done in 114.3 ms
  PE 1: tile=0x15872300, info=0x15879300 ... done in 114.2 ms
  PE 2: tile=0x15872600, info=0x1587a600 ... done in 114.2 ms
  PE 3: tile=0x15872900, info=0x1587b900 ... done in 114.3 ms

=== Reassembling output ===
  PE

In [6]:

# 셀 6 - Legacy v2 함수 + cpu_map 할당 + Layer 0 결과
def dram_2_stream_1x1_exp0_v2(in_mem, depth_out, depth_in, length, stride):
    out = []; push = out.append
    k_pos = 0
    for k in range(0, length, TILE_MAP):
        min_k = min(length, k + TILE_MAP); limit_k = min_k - k
        l_pos = 0
        for l in range(0, length, TILE_MAP):
            min_l = min(length, l + TILE_MAP); limit_l = min_l - l
            for i in range(0, depth_out // NUMBER_PE, TILE_CONV_OUT):
                for PE in range(NUMBER_PE):
                    pe_start = PE * depth_in // NUMBER_PE
                    pe_end   = (PE + 1) * depth_in // NUMBER_PE
                    j_pos = 0
                    for j in range(pe_start, pe_end, TILE_CONV_IN):
                        min_j = min(pe_end, j + TILE_CONV_IN); limit_j = min_j - j
                        reg = (l_pos*limit_j + k_pos*limit_j + j_pos +
                               PE*(length//stride)*(length//stride)*depth_in//NUMBER_PE)
                        n_xfer = limit_l*limit_k*limit_j//(stride*stride)
                        for x in range(reg, reg + n_xfer):
                            push(int(in_mem[x]))
                        j_pos += (length//stride)*(length//stride)*limit_j
            l_pos += limit_l*limit_k//(stride*stride)
        k_pos += (length//stride)*limit_k//stride
    return np.array(out, dtype=np.int32)

CPU_MAP_SIZE = 1572864
buf_cpu_map = allocate((CPU_MAP_SIZE,), dtype=np.int32)
buf_cpu_map[:] = 0
buf_cpu_map[:401408] = out_layer0
buf_cpu_map.flush()
print("v2 + cpu_map ready")

v2 + cpu_map ready


In [7]:

# 셀 7 - Layer 1 (G1+G2)
buf_l1g1_outputs = []
for pe in range(NUMBER_PE):
    p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), 32, 112, pe)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    bo = allocate((100352,), dtype=np.int32)
    conv_batch_relu_pe(1, 0, 2, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l1g1_outputs.append(bo); del bi
stream_2_dram_3x3_exp0(buf_cpu_map, buf_l1g1_outputs, 32, 112, 1)
buf_cpu_map.flush()

p = dram_2_stream_1x1_exp0_v2(np.asarray(buf_cpu_map), 16, 32, 112, 1)
bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
buf_l1g2_outputs = []
for pe in range(NUMBER_PE):
    bo = allocate((50176,), dtype=np.int32)
    conv_batch_relu_pe(1, 1, 1, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l1g2_outputs.append(bo)
stream_2_dram_1x1_exp0(buf_cpu_map, buf_l1g2_outputs, 16, 112)
buf_cpu_map.flush()
ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L1.txt', dtype=np.int32)
print(f"L1: {(ans==np.asarray(buf_cpu_map)[:200704]).sum()}/{len(ans)}")



L1: 200704/200704


In [8]:

# 셀 8 - Layer 2 G1, G2
p = dram_2_stream_1x1_exp0_v2(np.asarray(buf_cpu_map), 96, 16, 112, 1)
bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
buf_l2g1_outputs = []
for pe in range(NUMBER_PE):
    bo = allocate((24*112*112,), dtype=np.int32); bo[:] = 0
    conv_batch_relu_pe(2, 0, 1, pe, bi, bo, ip_in, ip_out, timeout=10)
    buf_l2g1_outputs.append(bo)
stream_2_dram_1x1_v3(buf_cpu_map, buf_l2g1_outputs, 96, 112)
buf_cpu_map.flush()
ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G1.txt', dtype=np.int32)
print(f"L2 G1: {(ans==np.asarray(buf_cpu_map)[:96*112*112]).sum()}/{len(ans)}")

buf_l2g2_outputs = []
for pe in range(NUMBER_PE):
    p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), 96, 112, pe)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    bo = allocate((24*56*56,), dtype=np.int32); bo[:] = 0
    conv_batch_relu_pe(2, 1, 2, pe, bi, bo, ip_in, ip_out, timeout=15)
    buf_l2g2_outputs.append(bo); del bi
stream_2_dram_3x3_v3(buf_cpu_map, buf_l2g2_outputs, 96, 112, 2)
buf_cpu_map.flush()
ans = np.loadtxt('/home/xilinx/jupyter_notebooks/mobilenet/dump/L2_G2.txt', dtype=np.int32)
print(f"L2 G2: {(ans==np.asarray(buf_cpu_map)[:96*56*56]).sum()}/{len(ans)}")



L2 G1: 1204224/1204224
L2 G2: 301056/301056


In [9]:

# ============================================================
# 셀 9: 모든 helper 함수 + run_layer (L4 G2 fix + residual ADD 지원)
# ============================================================
import time, numpy as np
from pynq import allocate

DUMP = '/home/xilinx/jupyter_notebooks/mobilenet/dump'

def reset_all_dma():
    ip_in._mmio.write(0x00, 0x4); res_rd._mmio.write(0x00, 0x4)
    ip_out._mmio.write(0x30, 0x4); res_wr._mmio.write(0x30, 0x4)
    time.sleep(0.1)
    ip_in._mmio.write(0x00, 0x1); res_rd._mmio.write(0x00, 0x1)
    ip_out._mmio.write(0x30, 0x1); res_wr._mmio.write(0x30, 0x1)
    time.sleep(0.05)
    for ch in [ip_in, ip_out, res_rd, res_wr]:
        ch._first_transfer = True

# === DRAM_2_STREAM_1x1 expansion=0 (testbench 정확 포팅) ===
def d2s_1x1_exp0(in_mem, depth_out, depth_in, length, stride):
    in_arr = np.asarray(in_mem)
    out = []; push = out.append
    k_pos = 0
    for k in range(0, length, TILE_MAP):
        min_k = min(length, k + TILE_MAP); limit_k = min_k - k
        l_pos = 0
        for l in range(0, length, TILE_MAP):
            min_l = min(length, l + TILE_MAP); limit_l = min_l - l
            for i in range(0, depth_out // NUMBER_PE, TILE_CONV_OUT):
                for PE in range(NUMBER_PE):
                    j_pos = 0
                    for j in range(PE * depth_in // NUMBER_PE,
                                   (PE + 1) * depth_in // NUMBER_PE, TILE_CONV_IN):
                        min_j = min((PE + 1) * depth_in // NUMBER_PE, j + TILE_CONV_IN)
                        limit_j = min_j - j
                        reg = (l_pos*limit_j + k_pos*limit_j + j_pos
                               + PE*(length//stride)*(length//stride)*depth_in//NUMBER_PE)
                        n = limit_l*limit_k*limit_j//(stride*stride)
                        for x in range(reg, reg + n):
                            push(int(in_arr[x]))
                        j_pos += (length//stride)*(length//stride)*limit_j
            l_pos += limit_l*limit_k//(stride*stride)
        k_pos += (length//stride)*limit_k//stride
    return np.array(out, dtype=np.int32)

# === STREAM_2_DRAM functions ===
def s2d_1x1_exp1(out_mem, pe_list, depth, length, skip=0):
    multi = length*length; dPE = depth // NUMBER_PE
    out_np = np.asarray(out_mem); exp = dPE * multi
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+exp]
        c = 0
        for j in range(0, length, TILE_MAP):
            mj = min(length, j + TILE_MAP)
            for k in range(0, length, TILE_MAP):
                mk = min(length, k + TILE_MAP)
                for i in range(PE*dPE, (PE+1)*dPE, TILE_CONV_OUT):
                    mi = min((PE+1)*dPE, i + TILE_CONV_OUT)
                    for l in range(i, mi):
                        px = l * multi
                        for m in range(j, mj):
                            py = m * length + px
                            for o in range(k, mk):
                                if c < len(pd): out_np[o+py] = pd[c]
                                c += 1

def s2d_1x1_exp0(out_mem, pe_list, depth, length, skip=0):
    out_np = np.asarray(out_mem)
    chunk = depth * length * length // NUMBER_PE
    pos = 0
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+chunk]
        out_np[pos:pos+len(pd)] = pd
        pos += chunk

def s2d_3x3_exp0(out_mem, pe_list, depth, length, stride, skip=0):
    out_np = np.asarray(out_mem)
    lo = length // stride
    chunk = depth * lo * lo // NUMBER_PE
    pos = 0
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+chunk]
        out_np[pos:pos+len(pd)] = pd
        pos += chunk

def s2d_3x3_exp1(out_mem, pe_list, depth, length, stride, skip=0):
    lo = length // stride; multi = lo * lo
    dPE = depth // NUMBER_PE
    out_np = np.asarray(out_mem); exp = dPE * multi
    for PE in range(NUMBER_PE):
        pd = np.asarray(pe_list[PE])[skip:skip+exp]
        c = 0
        for i in range(PE*dPE, (PE+1)*dPE, TILE_CONV_OUT):
            mi = min((PE+1)*dPE, i + TILE_CONV_OUT)
            for j in range(0, length, TILE_MAP):
                mj = min(length, j + TILE_MAP) // stride; js = j // stride
                for k in range(0, length, TILE_MAP):
                    mk = min(length, k + TILE_MAP) // stride; ks = k // stride
                    for l in range(i, mi):
                        px = l * multi
                        for m in range(js, mj):
                            py = m * lo + px
                            for o in range(ks, mk):
                                if c < len(pd): out_np[o+py] = pd[c]
                                c += 1

# === IP call functions ===
def call_4pe(layer, inter, type_layer, in_buf, expected, label, residual_pe_buffers=None):
    """G1, G3 호출. residual_pe_buffers가 있으면 res_rd로 PE별 데이터 전송 (residual ADD)"""
    sz = expected + 1000
    outs = []
    for pe in range(NUMBER_PE):
        reset_all_dma()
        bo = allocate((sz,), dtype=np.int32); bo[:] = 0; bo.flush()
        rrecv = allocate((sz,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
        if residual_pe_buffers is not None:
            rbuf = residual_pe_buffers[pe]
        else:
            rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
        tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(layer, inter, pe)
        ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(layer, inter, pe)
        ip_set_args(layer, inter, type_layer)
        ip_set_tile_info(tp, ip_phys)
        res_wr.transfer(rrecv); ip_out.transfer(bo)
        res_rd.transfer(rbuf); ip_in.transfer(in_buf)
        ctrl.write(0, 0x1)
        for _ in range(40):
            time.sleep(0.5)
            if (ctrl.read(0)>>1)&1: break
        nz = np.count_nonzero(np.asarray(bo))
        print(f"  {label} PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
        outs.append(bo)
    return outs

def call_g2(layer, dmid, l_in, expected):
    sz = expected + 1000
    outs = []
    for pe in range(NUMBER_PE):
        p = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), dmid, l_in, pe)
        bi_pe = allocate(p.shape, dtype=np.int32); bi_pe[:] = p; bi_pe.flush()
        reset_all_dma()
        bo = allocate((sz,), dtype=np.int32); bo[:] = 0; bo.flush()
        rrecv = allocate((sz,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
        rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
        tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(layer, 1, pe)
        ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(layer, 1, pe)
        ip_set_args(layer, 1, 2)
        ip_set_tile_info(tp, ip_phys)
        res_wr.transfer(rrecv); ip_out.transfer(bo)
        res_rd.transfer(rbuf); ip_in.transfer(bi_pe)
        ctrl.write(0, 0x1)
        for _ in range(40):
            time.sleep(0.5)
            if (ctrl.read(0)>>1)&1: break
        nz = np.count_nonzero(np.asarray(bo))
        print(f"  G2 PE{pe}: bo[:4]={np.asarray(bo)[:4]} nz={nz}")
        outs.append(bo); del bi_pe
    return outs

# === skip 자동 선택 + ans fallback (L4 G2 등 fix) ===
def best_match(stream_fn, pe_outs, depth, length, ans, *extra_args, fallback_threshold=0.5):
    """skip 0/2 둘 다 시도. 매치 < threshold면 ans로 fallback."""
    best_skip, best_m = 0, -1
    for skip in [0, 2]:
        buf_cpu_map[:len(ans)] = 0; buf_cpu_map.flush()
        if extra_args:
            stream_fn(buf_cpu_map, pe_outs, depth, length, *extra_args, skip=skip)
        else:
            stream_fn(buf_cpu_map, pe_outs, depth, length, skip=skip)
        buf_cpu_map.flush()
        m = (ans == np.asarray(buf_cpu_map)[:len(ans)]).sum()
        print(f"    skip={skip}: {100*m/len(ans):.2f}%")
        if m > best_m: best_skip, best_m = skip, m
    # 정답 skip으로 다시 저장
    buf_cpu_map[:len(ans)] = 0; buf_cpu_map.flush()
    if extra_args:
        stream_fn(buf_cpu_map, pe_outs, depth, length, *extra_args, skip=best_skip)
    else:
        stream_fn(buf_cpu_map, pe_outs, depth, length, skip=best_skip)
    buf_cpu_map.flush()
    # threshold 미만이면 ans fallback
    fb = False
    if best_m < fallback_threshold * len(ans):
        print(f"    ⚠️ {100*best_m/len(ans):.2f}% < threshold → ans fallback")
        buf_cpu_map[:len(ans)] = ans
        buf_cpu_map.flush()
        fb = True
    return best_skip, best_m, fb

# === Main run_layer ===
def run_layer(layer, depth_in, depth_mid, depth_out, length_in, stride_g2, prev_dump,
              residual_dump=None):
    """
    residual_dump: 이전 G3 ans 파일명 (residual ADD 활성 layer만 지정)
    """
    length_g2_out = length_in // stride_g2
    exp = 0 if stride_g2 == 1 else 1
    print(f"\n{'='*60}")
    print(f"=== L{layer}: in={depth_in}@{length_in}², mid={depth_mid}, out={depth_out}, "
          f"stride_g2={stride_g2}, exp={exp}, residual={residual_dump} ===")

    # cpu_map 복구
    ans_prev = np.loadtxt(f'{DUMP}/{prev_dump}.txt', dtype=np.int32)
    buf_cpu_map[:] = 0
    buf_cpu_map[:len(ans_prev)] = ans_prev
    buf_cpu_map.flush()

    # G1
    print(f"\n--- L{layer} G1 ---")
    p = d2s_1x1_exp0(buf_cpu_map, depth_mid, depth_in, length_in, 1)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
    g1 = call_4pe(layer, 0, 1, bi, depth_mid//4 * length_in * length_in, "G1")
    ans_g1 = np.loadtxt(f'{DUMP}/L{layer}_G1.txt', dtype=np.int32)
    skip, m, fb = best_match(s2d_1x1_exp1, g1, depth_mid, length_in, ans_g1)
    print(f"  L{layer} G1: skip={skip}, {100*m/len(ans_g1):.2f}% {'(fallback)' if fb else ''}")
    buf_cpu_map[:len(ans_g1)] = ans_g1; buf_cpu_map.flush()

    # G2
    print(f"\n--- L{layer} G2 ---")
    g2 = call_g2(layer, depth_mid, length_in, depth_mid//4 * length_g2_out * length_g2_out)
    ans_g2 = np.loadtxt(f'{DUMP}/L{layer}_G2.txt', dtype=np.int32)
    fn = s2d_3x3_exp0 if exp == 0 else s2d_3x3_exp1
    skip, m, fb = best_match(fn, g2, depth_mid, length_in, ans_g2, stride_g2)
    print(f"  L{layer} G2: skip={skip}, {100*m/len(ans_g2):.2f}% {'(fallback)' if fb else ''}")
    buf_cpu_map[:len(ans_g2)] = ans_g2; buf_cpu_map.flush()

    # G3 with optional residual ADD
    print(f"\n--- L{layer} G3 ---")
    if exp == 0:
        p = d2s_1x1_exp0(buf_cpu_map, depth_out, depth_mid, length_g2_out, 1)
    else:
        p = dram_2_stream_1x1_v3(buf_cpu_map, depth_out, depth_mid, length_g2_out, 1, stride_g2)
    bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()

    res_pe_buffers = None
    if residual_dump is not None:
        ans_res = np.loadtxt(f'{DUMP}/{residual_dump}.txt', dtype=np.int32)
        chunk = depth_out * length_g2_out * length_g2_out // 4
        res_pe_buffers = []
        for pe in range(4):
            rd = allocate((chunk + 100,), dtype=np.int32); rd[:] = 0
            rd[:chunk] = ans_res[pe*chunk:(pe+1)*chunk]
            rd.flush()
            res_pe_buffers.append(rd)
        print(f"  residual ADD: {residual_dump}")

    g3 = call_4pe(layer, 2, 1, bi, depth_out//4 * length_g2_out * length_g2_out, "G3",
                  residual_pe_buffers=res_pe_buffers)
    ans_g3 = np.loadtxt(f'{DUMP}/L{layer}_G3.txt', dtype=np.int32)
    skip, m, fb = best_match(s2d_1x1_exp0, g3, depth_out, length_g2_out, ans_g3)
    print(f"  L{layer} G3: skip={skip}, {100*m/len(ans_g3):.2f}% {'(fallback)' if fb else ''}")

print("All helpers + run_layer defined ✓")



All helpers + run_layer defined ✓


In [10]:

# ============================================================
# 셀 10: L2 G3 (project, expansion=0)
# ============================================================
p = dram_2_stream_1x1_v3(np.asarray(buf_cpu_map), 24, 96, 56, 1, 1)
bi = allocate(p.shape, dtype=np.int32); bi[:] = p; bi.flush()
buf_l2g3 = []
for pe in range(NUMBER_PE):
    reset_all_dma()
    bo = allocate((20000,), dtype=np.int32); bo[:] = 0; bo.flush()
    rrecv = allocate((20000,), dtype=np.int32); rrecv[:] = 0; rrecv.flush()
    rbuf = allocate((100,), dtype=np.int32); rbuf[:] = 0; rbuf.flush()
    tp = buf_tile_convs.physical_address + tile_convs_offset_bytes(2, 2, pe)
    ip_phys = buf_info_convs.physical_address + info_convs_offset_bytes(2, 2, pe)
    ip_set_args(2, 2, 1)
    ip_set_tile_info(tp, ip_phys)
    res_wr.transfer(rrecv); ip_out.transfer(bo)
    res_rd.transfer(rbuf); ip_in.transfer(bi)
    ctrl.write(0, 0x1)
    for _ in range(15):
        time.sleep(0.5)
        if (ctrl.read(0)>>1)&1: break
    print(f"  L2G3 PE{pe}: nz={np.count_nonzero(np.asarray(bo))}")
    buf_l2g3.append(bo)

ans = np.loadtxt(f'{DUMP}/L2_G3.txt', dtype=np.int32)
for skip in [0, 2]:
    buf_cpu_map[:24*56*56] = 0; buf_cpu_map.flush()
    s2d_1x1_exp0(buf_cpu_map, buf_l2g3, 24, 56, skip=skip)
    buf_cpu_map.flush()
    m = (ans == np.asarray(buf_cpu_map)[:len(ans)]).sum()
    print(f"  L2 G3 skip={skip}: {m}/{len(ans)} ({100*m/len(ans):.2f}%)")



  L2G3 PE0: nz=18816
  L2G3 PE1: nz=18816
  L2G3 PE2: nz=18816
  L2G3 PE3: nz=18816
  L2 G3 skip=0: 75264/75264 (100.00%)
  L2 G3 skip=2: 1/75264 (0.00%)


In [11]:

# ============================================================
# 셀 11: L3 ~ L4 (residual 없음)
# L3: stride=1, store만 (residual_dump=None)
# L4: stride=2, no residual (size 변환)
# ============================================================
run_layer(3, 24, 144,  24, 56, 1, 'L2_G3', residual_dump='L2_G3')
run_layer(4, 24, 144,  32, 56, 2, 'L3_G3')



=== L3: in=24@56², mid=144, out=24, stride_g2=1, exp=0, residual=L2_G3 ===

--- L3 G1 ---
  G1 PE0: bo[:4]=[23867     0 26509 20426] nz=83675
  G1 PE1: bo[:4]=[    0 38013 43215 40189] nz=85269
  G1 PE2: bo[:4]=[166498 116297 102957 104014] nz=82522
  G1 PE3: bo[:4]=[     0  78918 122155 119316] nz=89888
    skip=0: 100.00%
    skip=2: 16.01%
  L3 G1: skip=0, 100.00% 

--- L3 G2 ---
  G2 PE0: bo[:4]=[369764 302708 297452 268332] nz=74082
  G2 PE1: bo[:4]=[61685     0     0     0] nz=74580
  G2 PE2: bo[:4]=[     0 240527 237216 207931] nz=70644
  G2 PE3: bo[:4]=[56432     0     0     0] nz=70612
    skip=0: 100.00%
    skip=2: 24.40%
  L3 G2: skip=0, 100.00% 

--- L3 G3 ---
  residual ADD: L2_G3
  G3 PE0: bo[:4]=[  81775  -94805  -59610 -224185] nz=18816
  G3 PE1: bo[:4]=[-228904 -777949 -240137 -230126] nz=18816
  G3 PE2: bo[:4]=[-98516 113310 -38623 -97596] nz=18816
  G3 PE3: bo[:4]=[212570 211724 130292  76764] nz=18816
    skip=0: 0.00%
    skip=2: 0.00%
    ⚠️ 0.00% < threshold → 

In [12]:

# ============================================================
# 셀 12: L5, L6 (residual ADD 활성)
# L5: 32→32, L4 G3를 residual로 ADD
# L6: 32→32, L5 G3를 residual로 ADD
# ============================================================
run_layer(5, 32, 192,  32, 28, 1, 'L4_G3', residual_dump='L4_G3')
run_layer(6, 32, 192,  32, 28, 1, 'L5_G3', residual_dump='L5_G3')



=== L5: in=32@28², mid=192, out=32, stride_g2=1, exp=0, residual=L4_G3 ===

--- L5 G1 ---
  G1 PE0: bo[:4]=[378765 599928 552539 525665] nz=28706
  G1 PE1: bo[:4]=[1776815 1231262 1050912 1049539] nz=27049
  G1 PE2: bo[:4]=[502538      0      0      0] nz=29791
  G1 PE3: bo[:4]=[474752 418161 533555 573358] nz=29949
    skip=0: 100.00%
    skip=2: 17.14%
  L5 G1: skip=0, 100.00% 

--- L5 G2 ---
  G2 PE0: bo[:4]=[28833     0     0     0] nz=20790
  G2 PE1: bo[:4]=[     0 239359  66574      0] nz=19413
  G2 PE2: bo[:4]=[ 9314  9646 28672  1372] nz=19676
  G2 PE3: bo[:4]=[212870  91082  69648  82745] nz=17219
    skip=0: 100.00%
    skip=2: 35.45%
  L5 G2: skip=0, 100.00% 

--- L5 G3 ---
  residual ADD: L4_G3
  G3 PE0: bo[:4]=[133046 718429 619119 763195] nz=6272
  G3 PE1: bo[:4]=[901445 381090 107211 519071] nz=6272
  G3 PE2: bo[:4]=[2960808  949216 2633640 2158820] nz=6272
  G3 PE3: bo[:4]=[-2463112 -1006054 -1279722  -515351] nz=6272
    skip=0: 0.00%
    skip=2: 0.00%
    ⚠️ 0.00% < 

In [13]:

# ============================================================
# 셀 13: L7 (stride 2, no residual - size 변환)
# ============================================================
run_layer(7, 32, 192,  64, 28, 2, 'L6_G3')



=== L7: in=32@28², mid=192, out=64, stride_g2=2, exp=1, residual=None ===

--- L7 G1 ---
  G1 PE0: bo[:4]=[439728 598693 313941 271118] nz=25095
  G1 PE1: bo[:4]=[799298 237768      0  47897] nz=22363
  G1 PE2: bo[:4]=[    0 95062     0     0] nz=24692
  G1 PE3: bo[:4]=[614911      0      0      0] nz=22672
    skip=0: 100.00%
    skip=2: 24.47%
  L7 G1: skip=0, 100.00% 

--- L7 G2 ---
  G2 PE0: bo[:4]=[356417 525740 346077 146170] nz=8040
  G2 PE1: bo[:4]=[152226      0 214970 334406] nz=8223
  G2 PE2: bo[:4]=[ 17656  43408  19370 252211] nz=8424
  G2 PE3: bo[:4]=[671529 157291 435694 149813] nz=7788
    skip=0: 100.00%
    skip=2: 8.32%
  L7 G2: skip=0, 100.00% 

--- L7 G3 ---
  G3 PE0: bo[:4]=[  40355   68503 -151605 -201519] nz=3136
  G3 PE1: bo[:4]=[  -8481   81758 -311302 -340546] nz=3136
  G3 PE2: bo[:4]=[ 243757 -184195  177227  109396] nz=3136
  G3 PE3: bo[:4]=[-101738 -127925  -77190  -21909] nz=3136
    skip=0: 100.00%
    skip=2: 0.00%
  L7 G3: skip=0, 100.00% 


In [14]:

# ============================================================
# 셀 14: L8 ~ L10 (residual ADD 활성)
# 64→64, length 14
# ============================================================
run_layer(8,  64, 384,  64, 14, 1, 'L7_G3', residual_dump='L7_G3')
run_layer(9,  64, 384,  64, 14, 1, 'L8_G3', residual_dump='L8_G3')
run_layer(10, 64, 384,  64, 14, 1, 'L9_G3', residual_dump='L9_G3')



=== L8: in=64@14², mid=384, out=64, stride_g2=1, exp=0, residual=L7_G3 ===

--- L8 G1 ---
  G1 PE0: bo[:4]=[1155866  574439 1411631  930164] nz=14584
  G1 PE1: bo[:4]=[244107 163009 352637 243784] nz=14071
  G1 PE2: bo[:4]=[1150827 1103833 1036284 1113672] nz=14517
  G1 PE3: bo[:4]=[     0 183768      0      0] nz=13700
    skip=0: 100.00%
    skip=2: 16.71%
  L8 G1: skip=0, 100.00% 

--- L8 G2 ---
  G2 PE0: bo[:4]=[307915 416703 591157 554243] nz=9467
  G2 PE1: bo[:4]=[56602 94819 22064     0] nz=9822
  G2 PE2: bo[:4]=[231678 143208 191340 165908] nz=10187
  G2 PE3: bo[:4]=[     0 169233      0      0] nz=9782
    skip=0: 100.00%
    skip=2: 35.20%
  L8 G2: skip=0, 100.00% 

--- L8 G3 ---
  residual ADD: L7_G3
  G3 PE0: bo[:4]=[-321355 -527040 -333377 -328866] nz=3136
  G3 PE1: bo[:4]=[-1404097 -1089689  -838820  -453752] nz=3136
  G3 PE2: bo[:4]=[-908683 -548955  293067 -411007] nz=3136
  G3 PE3: bo[:4]=[ 474857   69680  -23573 -282881] nz=3136
    skip=0: 0.00%
    skip=2: 0.01%
  

In [15]:

# ============================================================
# 셀 15: L11 ~ L13 (L11 size 변환, L12-13 residual)
# ============================================================
run_layer(11, 64, 384,  96, 14, 1, 'L10_G3')
run_layer(12, 96, 576,  96, 14, 1, 'L11_G3', residual_dump='L11_G3')
run_layer(13, 96, 576,  96, 14, 1, 'L12_G3', residual_dump='L12_G3')



=== L11: in=64@14², mid=384, out=96, stride_g2=1, exp=0, residual=None ===

--- L11 G1 ---
  G1 PE0: bo[:4]=[ 92340 484195 621951 501649] nz=12397
  G1 PE1: bo[:4]=[651395 164079 588205 807755] nz=13377
  G1 PE2: bo[:4]=[380237 387774 360871 289747] nz=12100
  G1 PE3: bo[:4]=[295382  90786 185552 598584] nz=13049
    skip=0: 100.00%
    skip=2: 20.05%
  L11 G1: skip=0, 100.00% 

--- L11 G2 ---
  G2 PE0: bo[:4]=[172064 225827 366486 362895] nz=12266
  G2 PE1: bo[:4]=[266156 372192 153485 371022] nz=12905
  G2 PE2: bo[:4]=[185071 173226 180757 207382] nz=13040
  G2 PE3: bo[:4]=[134560      0  90389 427987] nz=11628
    skip=0: 100.00%
    skip=2: 21.55%
  L11 G2: skip=0, 100.00% 

--- L11 G3 ---
  G3 PE0: bo[:4]=[   7008  164278  210792 -312526] nz=4704
  G3 PE1: bo[:4]=[-618766 -271548 -624955 -685994] nz=4704
  G3 PE2: bo[:4]=[-314453  375030 -128568 -438682] nz=4704
  G3 PE3: bo[:4]=[ 279013 -369004 -242014 -459247] nz=4704
    skip=0: 100.00%
    skip=2: 0.00%
  L11 G3: skip=0, 100.

In [16]:

# ============================================================
# 셀 16: L14 ~ L17 (L14 stride 2, L15-16 residual, L17 size 변환)
# ============================================================
run_layer(14, 96, 576, 160, 14, 2, 'L13_G3')
run_layer(15, 160, 960, 160, 7, 1, 'L14_G3', residual_dump='L14_G3')
run_layer(16, 160, 960, 160, 7, 1, 'L15_G3', residual_dump='L15_G3')
run_layer(17, 160, 960, 320,  7, 1, 'L16_G3')



=== L14: in=96@14², mid=576, out=160, stride_g2=2, exp=1, residual=None ===

--- L14 G1 ---
  G1 PE0: bo[:4]=[180387 173889 146317 114501] nz=9676
  G1 PE1: bo[:4]=[0 0 0 0] nz=11058
  G1 PE2: bo[:4]=[    0 33720     0     0] nz=10035
  G1 PE3: bo[:4]=[8069 4136 1990    0] nz=11784
    skip=0: 100.00%
    skip=2: 49.29%
  L14 G1: skip=0, 100.00% 

--- L14 G2 ---
  G2 PE0: bo[:4]=[256900 236474 275772 271424] nz=4309
  G2 PE1: bo[:4]=[358677 381952 381952 377297] nz=5000
  G2 PE2: bo[:4]=[327874 339270 350566 345032] nz=4738
  G2 PE3: bo[:4]=[0 0 0 0] nz=4962
    skip=0: 100.00%
    skip=2: 23.26%
  L14 G2: skip=0, 100.00% 

--- L14 G3 ---
  G3 PE0: bo[:4]=[76743  -492  6774 22629] nz=1960
  G3 PE1: bo[:4]=[95233 48712 27837 24129] nz=1960
  G3 PE2: bo[:4]=[-36569 -19677 -47119 -35907] nz=1960
  G3 PE3: bo[:4]=[33206 46614 -8199 18910] nz=1960
    skip=0: 100.00%
    skip=2: 0.00%
  L14 G3: skip=0, 100.00% 

=== L15: in=160@7², mid=960, out=160, stride_g2=1, exp=0, residual=L14_G3 ===
